In [114]:
from scipy.integrate import quad
from math import exp, inf, pi
from scipy.special import erf
import numpy as np
import scipy.stats as st
import scipy.optimize as opt

In [115]:
def gamma_func(alpha):
    return quad(lambda t: t ** (alpha - 1) * exp(-t), 0, inf)[0]

def chi2_density(df, val):
    return (1/2) ** (df/2) / gamma_func(df/2) * val ** (df/2 - 1) * exp(-val/2)

def chi2_survival(df, low, high):
    return quad(lambda x: chi2_density(df, x), low, high)[0]

In [116]:
all_results = []

### Раздел A. Проверка гипотезы о равномерном распределении (критерий $\chi^2$)

In [117]:
res_a_chi2 = {
    "test_name": "Uniform distribution, chi-square test",
    "delta_est": 16.4,
    "p_value": chi2_survival(9, 16.4, inf),
    "decision": "Fail to reject H0"
}
all_results.append(res_a_chi2)

### Раздел A. Критерий Колмогорова для равномерного распределения

In [118]:
def kolmogorov_cdf_func(z):
    s = 0
    for m in range(1, 10000):
        s += (-1) ** m * exp(-2 * m ** 2 * z ** 2)
    return 1 + 2 * s

In [119]:
def compute_delta_uniform(counts, total_n, n_intervals):
    unit_vec = np.ones(n_intervals)
    theo_cdf = np.array([unit_vec[:j].sum() for j in range(len(unit_vec))]) / 10
    emp_cdf_left = np.array([counts[:j].sum() for j in range(len(counts) + 1)]) / total_n

    max_diff_val = max(
        max(abs(emp_cdf_left[j] - theo_cdf[j]), abs(emp_cdf_left[j + 1] - theo_cdf[j])) 
        for j in range(n_intervals)
    )
    return total_n ** 0.5 * max_diff_val

In [120]:
freq_data = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])

N_total = 100
K_intervals = len(freq_data)
alpha_level = 0.05

delta_val = compute_delta_uniform(freq_data, N_total, K_intervals)
p_val_kolmogorov = 1 - kolmogorov_cdf_func(delta_val)

res_a_kolmogorov = {
    "test_name": "Uniform distribution, Kolmogorov test",
    "delta_est": delta_val,
    "p_value": p_val_kolmogorov,
    "decision": "Reject H0" if p_val_kolmogorov < alpha_level else "Fail to reject H0"
}
all_results.append(res_a_kolmogorov)

### Раздел A. Параметрический бутстрап для критерия Колмогорова (равномерное распределение)

In [121]:
B_unif = 50000
boot_stats_unif = []
for _ in range(B_unif):
    boot_sample = st.uniform.rvs(loc=0, scale=10, size=N_total)
    boot_delta = compute_delta_uniform(boot_sample, N_total, K_intervals)
    boot_stats_unif.append(boot_delta)

boot_stats_unif = np.array(sorted(boot_stats_unif))
p_val_boot_unif = np.mean(np.array(boot_stats_unif) >= delta_val) / B_unif

res_a_bootstrap = {
    "test_name": "Uniform distribution, Kolmogorov test (parametric bootstrap)",
    "p_value": p_val_boot_unif,
    "decision": "Reject H0" if p_val_boot_unif < alpha_level else "Fail to reject H0"
}
all_results.append(res_a_bootstrap)

### Раздел B. Проверка гипотезы о нормальном распределении (критерий $\chi^2$)

In [122]:
bounds_arr = np.arange(0.5, 11.5, 1)

def neg_log_lik_normal(params):
    mu, sigma = params
    if sigma <= 0:
        return 1e10
    
    probs_vals = np.diff(st.norm.cdf(bounds_arr, loc=mu, scale=sigma))
    probs_vals = np.maximum(probs_vals, 1e-15)
    probs_vals = probs_vals / np.sum(probs_vals)
    
    return -np.sum(freq_data * np.log(probs_vals))

opt_res = opt.differential_evolution(
    neg_log_lik_normal,
    bounds=[(0, 10), (0.1, 5)],
    maxiter=1000,
    popsize=15
)

mu_hat, sigma_hat = opt_res.x
print(f"MLE estimates: mu = {mu_hat:.4f}, sigma = {sigma_hat:.4f}")

def norm_pdf(x, mu, sigma):
    return 1 / (sigma * np.sqrt(2 * np.pi)) * exp(-((x - mu) ** 2) / (2 * sigma ** 2))

delta_chi2 = 0
intervals_list = [[-inf, 1], [1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7], [7, 8], [8, 9], [9, inf]]
for idx in range(1, 10):
    prob_i = quad(lambda x: norm_pdf(x, mu_hat, sigma_hat), intervals_list[idx][0], intervals_list[idx][1])[0]
    delta_chi2 += (freq_data[idx] - N_total * prob_i) ** 2 / (N_total * prob_i)
print(f"Delta estimate: {delta_chi2:.4f}")
p_val_chi2_norm = chi2_survival(9, delta_chi2, inf)
print(f"p-value = {p_val_chi2_norm:.4f}")

res_b_chi2 = {
    "test_name": "Normal distribution, chi-square test",
    "delta_est": delta_chi2,
    "p_value": p_val_chi2_norm,
    "decision": "Reject H0" if p_val_chi2_norm < alpha_level else "Fail to reject H0"
}
all_results.append(res_b_chi2)

MLE estimates: mu = 6.0291, sigma = 3.5018
Delta estimate: 20.6598
p-value = 0.0142


### Раздел B. Критерий Колмогорова для нормального распределения

In [123]:
def norm_cdf_func(x, mu, sigma):
    """
    Вычисление функции распределения нормального закона
    """
    if isinstance(x, np.ndarray):
        return 0.5 * (1 + erf((x - mu) / (np.sqrt(2) * sigma)))
    else:
        return 0.5 * (1 + erf((x - mu) / (np.sqrt(2) * sigma)))

In [124]:
def compute_delta_normal(counts, total_n, n_intervals, mu, sigma):
    points = np.arange(n_intervals)
    emp_cdf_left = np.array([counts[:j].sum() for j in range(len(counts) + 1)]) / total_n

    max_diff_val = max(
        max(abs(emp_cdf_left[j] - norm_cdf_func(points[j], mu, sigma)), 
            abs(emp_cdf_left[j + 1] - norm_cdf_func(points[j], mu, sigma))) 
        for j in range(n_intervals)
    )
    return total_n ** 0.5 * max_diff_val

In [125]:
delta_norm = compute_delta_normal(freq_data, N_total, K_intervals, mu_hat, sigma_hat)
p_val_kolmogorov_norm = 1 - kolmogorov_cdf_func(delta_norm)

res_b_kolmogorov = {
    "test_name": "Normal distribution, Kolmogorov test",
    "delta_est": delta_norm,
    "p_value": p_val_kolmogorov_norm,
    "decision": "Reject H0" if p_val_kolmogorov_norm < alpha_level else "Fail to reject H0"
}
all_results.append(res_b_kolmogorov)

### Раздел B. Параметрический бутстрап для критерия Колмогорова (нормальное распределение)

In [126]:
B_norm = 50000
boot_stats_norm = []
for _ in range(B_norm):
    boot_sample = st.norm.rvs(loc=mu_hat, scale=sigma_hat, size=N_total)
    
    mu_boot = boot_sample.mean()
    sigma_boot = boot_sample.std(ddof=1)

    boot_delta = compute_delta_normal(boot_sample, N_total, K_intervals, mu_boot, sigma_boot)
    boot_stats_norm.append(boot_delta)

boot_stats_norm = np.array(sorted(boot_stats_norm))
p_val_boot_norm = np.mean(np.array(boot_stats_norm) >= delta_norm) / B_norm

res_b_bootstrap = {
    "test_name": "Normal distribution, Kolmogorov test (parametric bootstrap)",
    "p_value": p_val_boot_norm,
    "decision": "Reject H0" if p_val_boot_norm < alpha_level else "Fail to reject H0"
}
all_results.append(res_b_bootstrap)

In [127]:
for rec in all_results:
    for key, val in rec.items():
        if isinstance(val, float):
            print(f"{key}: {val}")
        else:
            print(f"{key}: {val}")
    print()

test_name: Uniform distribution, chi-square test
delta_est: 16.4
p_value: 0.058984030544418545
decision: Fail to reject H0

test_name: Uniform distribution, Kolmogorov test
delta_est: 1.4000000000000001
p_value: 0.039681879538114306
decision: Reject H0

test_name: Uniform distribution, Kolmogorov test (parametric bootstrap)
p_value: 1.9998e-05
decision: Reject H0

test_name: Normal distribution, chi-square test
delta_est: 20.659827071536952
p_value: 0.014249545711304022
decision: Reject H0

test_name: Normal distribution, Kolmogorov test
delta_est: 2.4557884846380156
p_value: 1.1552057688657413e-05
decision: Reject H0

test_name: Normal distribution, Kolmogorov test (parametric bootstrap)
p_value: 1.1646e-05
decision: Reject H0

